# API Helpers

This notebook contains helper functions for OpenAI API interactions, prompt building, and response parsing.

In [ ]:
import json
from typing import Any, Dict, List, Optional

# Import from config and models notebooks (run those cells first)
# from config import PROMPT_TEMPLATE, SYSTEM_PROMPT, get_openai_config
# from models import GeneratedListing, ProductInput
# from logger import logger

## Configuration (Copy from config.ipynb if running standalone)

In [ ]:
# Copy these values from config.ipynb if running this notebook standalone
PROMPT_TEMPLATE = """Analyze this product image and create an e-commerce listing.

Product Info:
- Name: {name}
- Category: {category}
- Brand: {brand}
- Color: {color}
- Price: {price} {currency}

Return ONLY valid JSON in this format:
{{
  "title": "Attractive product title",
  "description": "Compelling 80-120 word description",
  "features": ["feature1", "feature2", "feature3"],
  "tags": ["tag1", "tag2", "tag3"]
}}
"""

SYSTEM_PROMPT = "You are a professional e-commerce copywriter. Return only valid JSON."

def get_openai_config():
    return {
        "model": "gpt-4o",
        "max_tokens": 500,
        "temperature": 0.4,
        "timeout": 15,
    }

## Models (Copy from models.ipynb if running standalone)

In [ ]:
# Copy these classes from models.ipynb if running this notebook standalone
from pydantic import BaseModel
from typing import List, Optional

class GeneratedListing(BaseModel):
    title: str
    description: str
    features: List[str] = []
    tags: List[str] = []

class ProductInput(BaseModel):
    product_id: str
    product_name: str
    price: float
    currency: str = "USD"
    category: str
    gender: Optional[str] = None
    color: Optional[str] = None
    brand: Optional[str] = None
    image_url: Optional[str] = None
    image_base64: Optional[str] = None
    tags: Optional[List[str]] = None

## Logging Setup

In [ ]:
import logging

# Simple logger setup (replace with logger.ipynb content if available)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

## Prompt Building Functions

In [ ]:
def build_product_prompt(product: ProductInput) -> str:
    logger.debug("api_helpers.build_product_prompt: building prompt for product %s", product.product_id)
    try:
        return PROMPT_TEMPLATE.format(
            name=product.product_name,
            category=product.category,
            brand=product.brand or "N/A",
            color=product.color or "N/A",
            price=product.price,
            currency=product.currency,
        )
    except Exception as exc:
        logger.exception("api_helpers.build_product_prompt: failed to build prompt for product %s", product.product_id)
        raise RuntimeError(
            f"api_helpers.build_product_prompt: failed to build prompt for product {product.product_id}: {exc}"
        ) from exc

## Chat Message Building

In [ ]:
def build_chat_messages(prompt: str, image_b64: str) -> List[Dict[str, Any]]:
    logger.debug("api_helpers.build_chat_messages: building chat payload")
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
                },
            ],
        },
    ]

## OpenAI API Call Function

In [ ]:
def call_openai_chat_api(client: Any, messages: List[Dict[str, Any]], config: Dict[str, Any]) -> str:
    if client is None:
        logger.error("api_helpers.call_openai_chat_api: OpenAI client is not initialized")
        raise RuntimeError("api_helpers.call_openai_chat_api: OpenAI client is not initialized")

    try:
        logger.info("api_helpers.call_openai_chat_api: sending OpenAI request")
        response = client.chat.completions.create(
            model=config["model"],
            messages=messages,
            max_tokens=config["max_tokens"],
            temperature=config["temperature"],
        )
        return response.choices[0].message.content
    except Exception as exc:
        logger.exception("api_helpers.call_openai_chat_api: OpenAI API request failed")
        raise RuntimeError(
            f"api_helpers.call_openai_chat_api: OpenAI API request failed: {exc}"
        ) from exc

## Response Parsing Functions

In [ ]:
def clean_markdown_code_blocks(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    return cleaned.strip()

In [ ]:
def parse_json_response(text: str) -> dict:
    cleaned = clean_markdown_code_blocks(text)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as exc:
        logger.exception("api_helpers.parse_json_response: failed to parse JSON response")
        raise RuntimeError(
            f"api_helpers.parse_json_response: failed to parse JSON response: {exc}"
        ) from exc

In [ ]:
def validate_listing_response(data: dict) -> tuple[bool, Optional[str]]:
    try:
        GeneratedListing(**data)
        return True, None
    except Exception as exc:
        return False, f"api_helpers.validate_listing_response: invalid listing response: {exc}"

## Test API Helpers

In [ ]:
# Test prompt building
test_product = ProductInput(
    product_id="test-123",
    product_name="Test Product",
    price=29.99,
    category="Electronics",
    brand="Test Brand",
    color="Blue"
)

prompt = build_product_prompt(test_product)
print("Generated prompt:")
print(prompt)

# Test JSON parsing
test_json = '```json\n{"title": "Test Title", "description": "Test description", "features": ["feature1"], "tags": ["tag1"]}\n```'
parsed = parse_json_response(test_json)
print("\nParsed JSON:")
print(parsed)

# Test validation
is_valid, error = validate_listing_response(parsed)
print(f"\nValidation result: {is_valid}")
if error:
    print(f"Error: {error}")